In [1]:
import torch
import numpy as np

from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.utils import train_test_split_edges
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, average_precision_score

from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

# Isti seed kao u GAE/VGAE notebook-ovima — istih train/val/test grane
# pa je poređenje sa neuronskim modelima pošteno
torch.manual_seed(42)
np.random.seed(42)

In [2]:
dataset = Planetoid(
    root='./data',
    name='Cora',          
    transform=NormalizeFeatures()
)

data = dataset[0]

# split edge-ova
data = train_test_split_edges(data)

# Sve postojeće pozitivne grane — za korektno negativno uzorkovanje
all_pos_edge_index = torch.cat(
    [data.train_pos_edge_index, data.val_pos_edge_index, data.test_pos_edge_index],
    dim=1,
)

/tmp/ipykernel_17909/2721847594.py:10: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data)


In [3]:
#negativni test edge-ovi
# Isključujemo SVE prave grane (train+val+test) iz negativnog uzorkovanja
test_neg = negative_sampling(
    edge_index=all_pos_edge_index,
    num_nodes=data.num_nodes,
    num_neg_samples=data.test_pos_edge_index.shape[1]
)

In [4]:
#pravljenje adjency liste
adj = [set() for _ in range(data.num_nodes)]

edge_index = data.train_pos_edge_index

for i in range(edge_index.shape[1]):

    u = edge_index[0, i].item()
    v = edge_index[1, i].item()

    adj[u].add(v)
    adj[v].add(u)

In [5]:
#common neighbors scoer 
def common_neighbors_score(u, v):

    return len(adj[u].intersection(adj[v]))

In [6]:
#predikcije score-ova
def predict_scores_with_pairs(pos_edge_index, neg_edge_index):
    
    scores = []
    labels = []
    pairs  = []

    # pozitivni
    for i in range(pos_edge_index.shape[1]):
        u = pos_edge_index[0, i].item()
        v = pos_edge_index[1, i].item()

        scores.append(common_neighbors_score(u, v))
        labels.append(1)
        pairs.append((u, v))

    # negativni
    for i in range(neg_edge_index.shape[1]):
        u = neg_edge_index[0, i].item()
        v = neg_edge_index[1, i].item()

        scores.append(common_neighbors_score(u, v))
        labels.append(0)
        pairs.append((u, v))

    return np.array(scores), np.array(labels), pairs

In [7]:
#evaulacija
scores, labels, pairs = predict_scores_with_pairs(
    data.test_pos_edge_index,
    test_neg
)

auc = roc_auc_score(labels, scores)
ap  = average_precision_score(labels, scores)

print("\n=== COMMON NEIGHBORS BASELINE ===")
print(f"Dataset: {dataset.name}")
print(f"AUC: {auc:.4f}")
print(f"AP : {ap:.4f}")


=== COMMON NEIGHBORS BASELINE ===
Dataset: Cora
AUC: 0.7082
AP : 0.7071


In [8]:
threshold = 1  # bar 1 zajednicki sused

false_positives = []

for (u, v), s, y in zip(pairs, scores, labels):
    if s >= threshold and y == 0:
        false_positives.append((u, v, s))

false_negatives = []

for (u, v), s, y in zip(pairs, scores, labels):
    if s < threshold and y == 1:
        false_negatives.append((u, v, s))

In [9]:
#najvise laznih pozitvnih
false_positives_sorted = sorted(false_positives, key=lambda x: -x[2])

print("\n=== TOP FALSE POSITIVES (Common Neighbors) ===")
for u, v, s in false_positives_sorted[:10]:
    print(f"{u} - {v} | CN={s}")


=== TOP FALSE POSITIVES (Common Neighbors) ===
1869 - 1107 | CN=1
112 - 126 | CN=1
1651 - 1804 | CN=1


In [10]:
false_negatives_sorted = sorted(false_negatives, key=lambda x: x[2])

print("\n=== TOP FALSE NEGATIVES (Common Neighbors) ===")
for u, v, s in false_negatives_sorted[:10]:
    print(f"{u} - {v} | CN={s}")


=== TOP FALSE NEGATIVES (Common Neighbors) ===
1516 - 2612 | CN=0
1757 - 2451 | CN=0
1986 - 1990 | CN=0
1351 - 1592 | CN=0
180 - 2037 | CN=0
171 - 775 | CN=0
903 - 2030 | CN=0
2281 - 2405 | CN=0
1012 - 1797 | CN=0
1050 - 1569 | CN=0


In [11]:
dataset = Planetoid(
    root='./data',
    name='CiteSeer',          
    transform=NormalizeFeatures()
)

data = dataset[0]

# split edge-ova
data = train_test_split_edges(data)

# Sve postojeće pozitivne grane — za korektno negativno uzorkovanje
all_pos_edge_index = torch.cat(
    [data.train_pos_edge_index, data.val_pos_edge_index, data.test_pos_edge_index],
    dim=1,
)

/tmp/ipykernel_17909/143216211.py:10: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data)


In [12]:
#negativni test edge-ovi
# Isključujemo SVE prave grane (train+val+test) iz negativnog uzorkovanja
test_neg = negative_sampling(
    edge_index=all_pos_edge_index,
    num_nodes=data.num_nodes,
    num_neg_samples=data.test_pos_edge_index.shape[1]
)

In [13]:
#pravljenje adjency liste
adj = [set() for _ in range(data.num_nodes)]

edge_index = data.train_pos_edge_index

for i in range(edge_index.shape[1]):

    u = edge_index[0, i].item()
    v = edge_index[1, i].item()

    adj[u].add(v)
    adj[v].add(u)

In [14]:
#predikcije score-ova
def predict_scores(pos_edge_index, neg_edge_index):

    scores = []
    labels = []

    # pozitivni edge-ovi
    for i in range(pos_edge_index.shape[1]):

        u = pos_edge_index[0, i].item()
        v = pos_edge_index[1, i].item()

        score = common_neighbors_score(u, v)

        scores.append(score)
        labels.append(1)

    # negativni edge-ovi
    for i in range(neg_edge_index.shape[1]):

        u = neg_edge_index[0, i].item()
        v = neg_edge_index[1, i].item()

        score = common_neighbors_score(u, v)

        scores.append(score)
        labels.append(0)

    return np.array(scores), np.array(labels)

In [15]:
#evaulacija
scores, labels = predict_scores(
    data.test_pos_edge_index,
    test_neg
)

auc = roc_auc_score(labels, scores)
ap  = average_precision_score(labels, scores)

print("\n=== COMMON NEIGHBORS BASELINE ===")
print(f"Dataset: {dataset.name}")
print(f"AUC: {auc:.4f}")
print(f"AP : {ap:.4f}")


=== COMMON NEIGHBORS BASELINE ===
Dataset: CiteSeer
AUC: 0.6538
AP : 0.6538
